# Plotting: PPO basic

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [104]:
import json
import pickle
import pandas as pd
import seaborn as sns
import tensorboard as tb
from matplotlib import pyplot as plt
from scipy import stats
from pathlib import Path
from math import degrees
import tikzplotlib
from matplotlib.lines import Line2D

## Learning log

In [7]:
run_path = Path(r"/home/jovyan/trajectory_optimization_1/work/max_altitude/results/ppo_basic/PPO_LauncherV1SubOrbital_60644_00000_0_2021-05-17_15-13-04")

In [8]:
datas = []

with (run_path / "result.json").open("r") as f:
    try:
        while True:            
            data = json.loads(f.readline())
            datas.append(data)
    except (EOFError, json.JSONDecodeError):
        pass
    
results = pd.DataFrame(datas)

In [9]:
results = pd.read_csv(run_path / "progress.csv")

In [10]:
results.keys()

Index(['episode_reward_max', 'episode_reward_min', 'episode_reward_mean',
       'episode_len_mean', 'episodes_this_iter', 'num_healthy_workers',
       'timesteps_total', 'agent_timesteps_total', 'done', 'episodes_total',
       'training_iteration', 'experiment_id', 'date', 'timestamp',
       'time_this_iter_s', 'time_total_s', 'pid', 'hostname', 'node_ip',
       'time_since_restore', 'timesteps_since_restore',
       'iterations_since_restore', 'trial_id', 'hist_stats/episode_reward',
       'hist_stats/episode_lengths', 'sampler_perf/mean_raw_obs_processing_ms',
       'sampler_perf/mean_inference_ms',
       'sampler_perf/mean_action_processing_ms',
       'sampler_perf/mean_env_wait_ms', 'sampler_perf/mean_env_render_ms',
       'timers/sample_time_ms', 'timers/sample_throughput',
       'timers/learn_time_ms', 'timers/learn_throughput',
       'timers/update_time_ms', 'info/num_steps_sampled',
       'info/num_agent_steps_sampled', 'info/num_steps_trained',
       'perf/cpu_ut

In [89]:
with open("./rewards.pkl", "rb") as f:
    idxs, initial_thetas, greedy_rewards = pickle.load(f)
    
greedy_rewards = greedy_rewards.squeeze()

pid_70 = 0.79368608
pid_80 = 0.83342516
pid_90 = 0.9014775

In [129]:
# plt.figure()
time_tot_s = results["time_total_s"].values[-1]
time_h = int(time_tot_s / 60 / 60)
time_m = int((time_tot_s - time_h * 60 * 60) / 60)
time_s = time_tot_s - (time_h * 60 * 60) - time_m * 60


(time_h, time_m, time_s)

(3, 50, 30.847928047180176)

## Plotting

Add this to the groupplot style in the resulting tikz to fix subplot title issue.
`, vertical sep=2cm`

In [34]:
fig_dir_path = Path("./figures")
fig_dir_path.mkdir(exist_ok=True)

In [120]:
plt.close('all')
plt.style.use("ggplot")

fig, axs = plt.subplots(2, 1, figsize=(8, 6))
results["episode_reward_mean"].plot(ax=axs[0], label="orginal")
results["episode_reward_mean"].rolling(50).mean().plot(label="moving average", ax=axs[0])
# results["episode_reward_min"].plot(ax=axs[0])
# results["episode_reward_max"].plot(ax=axs[0])
axs[0].set_title("Mean reward")
axs[0].legend()

lines = [
    axs[1].plot([0.], [0.], c="k", ls="--", label="PID")[0],
    axs[1].plot([0.], [0.], c="k", label="Greedy agent")[0],
]
for i, theta in enumerate(reversed(initial_thetas)):
    if int(degrees(theta)) in [100, 110]:
        continue
    lines.append(axs[1].plot(idxs, greedy_rewards[:, i], label=fr"$\theta^E_{{t=0}}={degrees(theta):2.0f}^\circ$")[0])
axs[1].axhline(pid_70, ls="--", c="C2")
axs[1].axhline(pid_80, ls="--", c="C1")
axs[1].axhline(pid_90, ls="--", c="C0")
axs[1].set_title("Greedy agent and PID controller rewards")
axs[1].legend(handles=lines)
# results[key].rolling(10).mean().plot()
# plt.yscale("log")

tikzplotlib.save(fig_dir_path / "model_reward.tex", extra_axis_parameters=[
    'width=0.9\\textwidth',
    'height=0.4\\textwidth'
])


fig, axs = plt.subplots(3, 1, figsize=(8, 6))
key = "info/learner/default_policy/learner_stats/kl"
results[key].plot(label="original", ax=axs[0])
results[key].rolling(25).mean().plot(label="moving average", ax=axs[0])
axs[0].set_yscale("log")
axs[0].set_title("KL-divergence")
axs[0].legend()

key = "info/learner/default_policy/learner_stats/policy_loss"
abs(results[key]).plot(label="raw", ax=axs[1])
abs(results[key].rolling(25).mean()).plot(label="moving average", ax=axs[1])
axs[1].set_title("Policy loss")
axs[1].set_ylim(top=0.025, bottom=0.0)

key = "info/learner/default_policy/learner_stats/vf_loss"
abs(results[key]).plot(label="raw",  ax=axs[2])
abs(results[key].rolling(25).mean()).plot(label="moving average", ax=axs[2])
axs[2].set_title("Value function loss")
axs[2].set_ylim(top=0.005, bottom=0)

plt.tight_layout()

# tikzplotlib.clean_figure()
tikzplotlib.save(fig_dir_path / "model_losses.tex", extra_axis_parameters=[
    'width=0.9\\textwidth',
    'height=0.3\\textwidth'
])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## Greedy learning history

In [9]:
def get_idxs(run_path):
    yield from sorted(int(checkpoint_path.name[-6:]) for checkpoint_path in run_path.glob("checkpoint_*"))


def checkpoint_path(run_path, idx=None):
    if idx is None:
        idx = max(get_idxs(run_path))
    checkpoint_path = run_path / f"checkpoint_{idx:06}/checkpoint-{idx}"
    if checkpoint_path.exists():
        return checkpoint_path


def restore_trainer(load_run_path=None, idx=None):
    load_run_path = load_run_path or run_path
    if path := checkpoint_path(load_run_path, idx):
        trainer.restore(str(path))
        print(f"Restored: {path}")
    else:
        print(f"WARNING: Checkpoint not found {path}")

In [16]:
list(get_idxs(run_path))[::500]

[1, 1001, 2001]

In [ ]:
for idx in tqdm(list(get_idxs(run_path))[::1000]):
    print(idx)